# Credit Card Agreement Processing with BigQuery and Gemini 2.5 Flash

This notebook is designed to automate the extraction of key terms from Credit Card Agreement PDFs stored in Google Cloud Storage.

### Processing Workflow:
1. **Configuration**: Set project and bucket parameters.
2. **Environment Setup**: Enable APIs and establish BigQuery connections.
3. **IAM Permissions**: Grant necessary roles to the BigQuery connection.
4. **Data Processing**: Utilize Gemini 2.5 Flash to extract metadata and terms.
5. **Storage**: Save the results into a structured BigQuery table.

**Note**: To run this notebook, you need at least `Editor` and `IAM Admin` permissions on the project.

In [ ]:
# @title Configuration Form
# @markdown Fill in these parameters to set up the environment.

PROJECT_ID = "ndy-814-20251212174826" # @param {type:"string"}
GCS_SOURCE = "gs://credit_card_agreements" # @param {type:"string"}
REGION = "us-central1" # @param {type:"string"}
DATASET_ID = "cc_agreement_processing" # @param {type:"string"}
TABLE_ID = "cc_agreement_details" # @param {type:"string"}
CONNECTION_ID = "vertex-ai-connection-cc" # @param {type:"string"}

print(f"Project ID set to: {PROJECT_ID}")
print(f"GCS Source set to: {GCS_SOURCE}")
print(f"Dataset set to: {DATASET_ID}")

In [ ]:
# Authenticate and set project
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated successfully")
except ImportError:
    print("Running outside of Colab environment")

!gcloud config set project {PROJECT_ID}
!gcloud config set ai/region {REGION}

In [ ]:
# Enable requisite APIs
print("Enabling APIs... this may take a moment.")
!gcloud services enable \
    aiplatform.googleapis.com \
    bigqueryconnection.googleapis.com \
    storage.googleapis.com \
    bigquery.googleapis.com

print("APIs enabled successfully.")

In [ ]:
from google.cloud import bigquery
import json
import time
import subprocess

client = bigquery.Client(project=PROJECT_ID)

# Create Dataset if it doesn't exist
dataset_path = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_path)
dataset.location = REGION
try:
    client.create_dataset(dataset, exists_ok=True)
    print(f"Dataset {dataset_path} is ready.")
except Exception as e:
    print(f"Dataset status check: {e}")

# Create BigQuery Connection
print("Creating/Verifying Connection...")
!bq mk --connection --location={REGION} --project_id={PROJECT_ID} --connection_type=CLOUD_RESOURCE {CONNECTION_ID} || echo "Note: Connection might already exist."

# Get the Service Account from the connection
try:
    # Use dot separator instead of colon for the connection resource path
    cmd = f"bq show --connection --location={REGION} --format=json {PROJECT_ID}.{REGION}.{CONNECTION_ID}"
    result = subprocess.check_output(cmd, shell=True)
    connection_info = json.loads(result)
    service_account = connection_info['cloudResource']['serviceAccountId']
    print(f"Connection Service Account: {service_account}")

    # Grant Vertex AI User role
    print(f"Granting Vertex AI User to {service_account}...")
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member="serviceAccount:{service_account}" \
        --role="roles/aiplatform.user" --quiet

    # Grant Storage Object Viewer role
    print(f"Granting Storage Object Viewer to {service_account}...")
    !gcloud projects add-iam-policy-binding {PROJECT_ID} \
        --member="serviceAccount:{service_account}" \
        --role="roles/storage.objectViewer" --quiet

    print("IAM bindings updated. Waiting 60 seconds for propagation...")
    time.sleep(60)
    print("Wait complete. Proceeding with model creation.")
except Exception as e:
    print(f"Error during setup: {e}")
    print("Please verify your permissions (IAM Admin needed).")


In [ ]:
# Creating model in BigQuery via Python Client
sql_model = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.{REGION}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'gemini-2.5-flash');
"""

print("Creating Gemini 2.5 Flash model in BigQuery...")
job = client.query(sql_model)
job.result()
print("Model created.")

In [ ]:
# Creating external table in BigQuery for GCS PDF files
# This handles the recursive folder structure automatically
sql_ext = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.{DATASET_ID}.cc_agreements_ext`
WITH CONNECTION `{PROJECT_ID}.{REGION}.{CONNECTION_ID}`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['{GCS_SOURCE}/*.pdf']
);
"""

print("Creating external table to index files...")
job = client.query(sql_ext)
job.result()
print("External table created.")

In [ ]:
# Perform analysis and create the destination table
print("Performing analysis with Gemini 2.5 Flash... this may take some time depending on the number of files.")

# ULTIMATE ROBUST EXTRACTION:
# 1. Very strict prompt demanding JSON and forbidding conversational text.
# 2. Multiline-aware Regex: (?s) allows the '.' to match newlines, which is
#    critical since model output often spans multiple lines.
# 3. Extraction of anything between the first '{' and last '}'.

sql_analyze = f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS
WITH raw_results AS (
  SELECT
    uri AS agreement_file_name,
    -- (?s) makes the dot match newlines, allowing us to capture multiline JSON blocks
    REGEXP_EXTRACT(ml_generate_text_llm_result, r'(?s)(\\{{.*\\}})') AS cleaned_json
  FROM
    ML.GENERATE_TEXT(
      MODEL `{PROJECT_ID}.{DATASET_ID}.gemini_flash_model`,
      TABLE `{PROJECT_ID}.{DATASET_ID}.cc_agreements_ext`,
      STRUCT(
        'Extract info from this Credit Card Agreement. Return ONLY a JSON object with these keys: bank_name, credit_card_name, document_length (number), interest_rate (string), late_fee (string), minimum_payment_due (string). Do not include any conversational text or markdown code blocks.' AS prompt,
        TRUE AS flatten_json_output
      )
    )
)
SELECT
  agreement_file_name,
  JSON_VALUE(cleaned_json, '$.bank_name') AS bank_name,
  JSON_VALUE(cleaned_json, '$.credit_card_name') AS credit_card_name,
  SAFE_CAST(JSON_VALUE(cleaned_json, '$.document_length') AS INT64) AS document_length,
  JSON_VALUE(cleaned_json, '$.interest_rate') AS `interest rate`,
  JSON_VALUE(cleaned_json, '$.late_fee') AS late_fee,
  JSON_VALUE(cleaned_json, '$.minimum_payment_due') AS minimum_payment_due
FROM
  raw_results;
"""

try:
    job = client.query(sql_analyze)
    job.result()
    print(f"Analysis complete. Results stored in {TABLE_ID}.")
except Exception as e:
    print(f"Error during analysis: {e}")
    print("If results are still NULL, verify the model output in Cell 7.")

In [ ]:
# Verify results
import pandas as pd
query_verify = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` LIMIT 10"
df = client.query(query_verify).to_dataframe()
df.head()